# Embeddings: 
A practical notebook to test embeddings technics and validate their developement on smaller databases before tackling the 3 millions papers from arXiv.

## Dense embeddings

In [2]:
from FlagEmbedding import BGEM3FlagModel
import duckdb
import pandas as pd
import numpy as np
from tqdm.notebook import tqdm
from torchao.quantization import Int8WeightOnlyConfig
from transformers import AutoModelForCausalLM, AutoTokenizer, TorchAoConfig

In [11]:
model = BGEM3FlagModel('BAAI/bge-m3', use_fp16=True, device='cuda:0', max_seq_len=2048, use_cache=True)

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 30 files:   0%|          | 0/30 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

In [ ]:
import duckdb
import pandas as pd

conn = duckdb.connect("data/arxiv_metadata.duckdb", read_only=True)

positives = conn.execute("""
    SELECT arxiv_id, title, abstract FROM papers
    WHERE (title ILIKE '%quantiz%' OR abstract ILIKE '%quantiz%')
      AND primary_category IN ('cs.LG', 'cs.CL', 'cs.AI', 'cs.CV')
    ORDER BY RANDOM() LIMIT 20
""").fetchdf()

noise = conn.execute("""
    SELECT arxiv_id, title, abstract FROM papers
    WHERE primary_category IN ('cs.LG', 'cs.CL', 'cs.AI', 'cs.CV')
    ORDER BY RANDOM() LIMIT 480
""").fetchdf()

conn.close()
sample = pd.concat([positives, noise], ignore_index=True).drop_duplicates("arxiv_id")
sample.head()

,arxiv_id,title,abstract
0,2505.09663,Analog Foundation Models,Analog in-memory computing (AIMC) is a promisi...
1,2502.10424,QuantSpec: Self-Speculative Decoding with Hier...,Large Language Models (LLMs) are increasingl...
2,2509.14061,Queen Detection in Beehives via Environmental ...,Queen bee presence is essential for the health...
3,2012.10939,Study of Energy-Efficient Distributed RLS-base...,"In this work, we present an energy-efficient..."
4,1907.01703,Don't take it lightly: Phasing optical random ...,In this paper we tackle the problem of recov...


In [ ]:
texts = (sample["title"] + ". " + sample["abstract"]).tolist()
doc_out = model.encode(texts, return_dense=True, return_sparse=False)
doc_dense = doc_out["dense_vecs"]   # shape: (n_docs, 1024)


pre tokenize: 100%|██████████| 2/2 [00:00<00:00, 47.30it/s]



Inference Embeddings: 100%|██████████| 2/2 [00:02<00:00,  1.47s/it]


In [9]:
import numpy as np

query = "quantization of neural network weights for efficient inference"
query_dense = model.encode([query], return_dense=True, return_sparse=False)["dense_vecs"][0]

scores = doc_dense @ query_dense   # dot product = cosine similarity here, because BGE-M3 vectors are already L2-normalized
top_idx = np.argsort(-scores)[:15]

for i in top_idx:
    print(f"{scores[i]:.3f}  {sample.iloc[i]['title']}")

NameError: name 'doc_dense' is not defined

## Dense + sparse embedding using BGE-M3

In [ ]:
conn = duckdb.connect("data/arxiv_metadata.duckdb", read_only=True)
corpus = conn.execute("""
    SELECT arxiv_id, title, abstract, submitted_date
    FROM papers
    WHERE primary_category IN ('cs.LG', 'cs.CL', 'cs.AI', 'cs.CV')
      AND submitted_date >= '2022-01-01'
    ORDER BY RANDOM()
    LIMIT 40000
""").fetchdf()
conn.close()

In [ ]:

texts = (corpus["title"] + ". " + corpus["abstract"]).tolist()
batch_size = 64

all_dense, all_sparse = [], []
for i in tqdm(range(0, len(texts), batch_size)):
    out = model.encode(texts[i:i+batch_size], return_dense=True, return_sparse=True)
    all_dense.append(out["dense_vecs"])
    all_sparse.extend(out["lexical_weights"])

doc_dense = np.concatenate(all_dense, axis=0)   # shape (N, 1024) — fixed size, one row per doc
# all_sparse stays a plain list of {token_id: weight} dicts — sparse vectors are ragged by nature,
# each doc only has weights for the terms it actually contains, so they can't stack into a matrix

  0%|          | 0/625 [00:00<?, ?it/s]

In [ ]:
q_out = model.encode([query], return_dense=True, return_sparse=True)
query_dense = q_out["dense_vecs"][0]
query_sparse = q_out["lexical_weights"][0]

dense_scores = doc_dense @ query_dense
sparse_scores = np.array([
    model.compute_lexical_matching_score(query_sparse, doc_sparse)
    for doc_sparse in all_sparse
])

In [ ]:
def rrf_fuse(*score_arrays, k=60):
    fused = np.zeros(len(score_arrays[0]))
    for scores in score_arrays:
        ranks = np.argsort(np.argsort(-scores))   # 0 = best in that channel
        fused += 1.0 / (k + ranks + 1)
    return fused

hybrid_scores = rrf_fuse(dense_scores, sparse_scores)
top_idx = np.argsort(-hybrid_scores)[:15]

for i in top_idx:
    print(f"hybrid={hybrid_scores[i]:.5f}  dense={dense_scores[i]:.3f}  sparse={sparse_scores[i]:.3f}  {corpus.iloc[i]['title']}")

hybrid=0.03252  dense=0.651  sparse=0.296  Vertical Layering of Quantized Neural Networks for Heterogeneous
  Inference
hybrid=0.03252  dense=0.666  sparse=0.289  Starting Positions Matter: A Study on Better Weight Initialization for Neural Network Quantization
hybrid=0.03016  dense=0.646  sparse=0.215  Neural Network Quantization with AI Model Efficiency Toolkit (AIMET)
hybrid=0.02942  dense=0.628  sparse=0.218  Quantification of Uncertainties in Probabilistic Deep Neural Network by
  Implementing Boosting of Variational Inference
hybrid=0.02725  dense=0.646  sparse=0.204  IDKM: Memory Efficient Neural Network Quantization via Implicit,
  Differentiable k-Means
hybrid=0.02715  dense=0.625  sparse=0.210  Stochastic Weight Sharing for Bayesian Neural Networks
hybrid=0.02712  dense=0.612  sparse=0.242  FineQuant: Unlocking Efficiency with Fine-Grained Weight-Only
  Quantization for LLMs
hybrid=0.02667  dense=0.619  sparse=0.213  Optimization of the quantization of dense neural networks f

# Evaluation set

Let's build teh evaluation set to measure the precisions and recall of the retrieval.

In [ ]:
conn = duckdb.connect("data/arxiv_metadata.duckdb", read_only=True)

# Broad net for positives — independent of hybrid's own ranking, so recall isn't measured circularly
likely_positive = conn.execute("""
    SELECT arxiv_id, title, abstract FROM papers
    WHERE (title ILIKE '%quantiz%' OR abstract ILIKE '%quantiz%')
      AND primary_category IN ('cs.LG','cs.CL','cs.AI','cs.CV')
    ORDER BY RANDOM() LIMIT 50
""").fetchdf()

# Hard negatives — the exact "quantification" confusion you just found in the wild
hard_negative = conn.execute("""
    SELECT arxiv_id, title, abstract FROM papers
    WHERE (title ILIKE '%quantif%' OR abstract ILIKE '%quantif%')
      AND title NOT ILIKE '%quantiz%' AND abstract NOT ILIKE '%quantiz%'
      AND primary_category IN ('cs.LG','cs.CL','cs.AI','cs.CV')
    ORDER BY RANDOM() LIMIT 15
""").fetchdf()

# Plain noise floor
random_negative = conn.execute("""
    SELECT arxiv_id, title, abstract FROM papers
    WHERE primary_category IN ('cs.LG','cs.CL','cs.AI','cs.CV')
    ORDER BY RANDOM() LIMIT 20
""").fetchdf()

conn.close()
to_label = pd.concat([likely_positive, hard_negative, random_negative], ignore_index=True).drop_duplicates("arxiv_id")

In [ ]:
to_label["label"] = pd.NA

for idx, row in to_label.iterrows():
    if pd.notna(to_label.at[idx, "label"]):
        continue
    print(f"\n[{idx}] {row['title']}\n{row['abstract'][:550]}...")
    ans = input("Relevant? (1 = about reducing numerical precision of weights/activations, 0 = not, q = stop): ")
    if ans == "q":
        break
    to_label.at[idx, "label"] = int(ans)

to_label.to_parquet("data/interim/quantization_golden_set.parquet")  # save once, so you never relabel


[0] Variational Inference with Latent Space Quantization for Adversarial
  Resilience
  Despite their tremendous success in modelling high-dimensional data
manifolds, deep neural networks suffer from the threat of adversarial attacks -
Existence of perceptually valid input-like samples obtained through careful
perturbation that lead to degradation in the performance of the underlying
model. Major concerns with existing defense mechanisms include
non-generalizability across different attacks, models and large inference time.
In this paper, we propose a generalized defense mechanism capitalizing on the
expressive power of regulari...

[1] Cross-Modal Discrete Representation Learning
  Recent advances in representation learning have demonstrated an ability to
represent information from different modalities such as video, text, and audio
in a single high-level embedding vector. In this work we present a
self-supervised learning framework that is able to learn a representation that
capture

In [ ]:
to_label.head()

In [ ]:

# Lire le fichier Parquet
df = pd.read_parquet("data/interim/quantization_golden_set.parquet", engine="pyarrow")

print(df.head(50))

      arxiv_id                                              title  \
0   1903.09940  Variational Inference with Latent Space Quanti...   
1   2106.05438       Cross-Modal Discrete Representation Learning   
2   2605.18856  SPHERICAL KV: Angle-Domain Attention and Rate-...   
3   2105.03536    Pareto-Optimal Quantized ResNet Is Mostly 4-bit   
4   1611.06342  Quantized neural network design under weight c...   
5   2412.10319  SCBench: A KV Cache-Centric Analysis of Long-C...   
6   2311.17752  BAND-2k: Banding Artifact Noticeable Database ...   
7   2510.25934  Robust GNN Watermarking via Implicit Perceptio...   
8   2202.02361  A Fast Network Exploration Strategy to Profile...   
9   2409.00592  Hyper-Compression: Model Compression via Hyper...   
10  2009.13108  NITI: Training Integer Neural Networks Using I...   
11  2602.00135  LLaVA-FA: Learning Fourier Approximation for C...   
12  2502.01330  Accelerating Linear Recurrent Neural Networks ...   
13  2505.17856  Stochastic Weight 

# Measuring recall and precision 

Let's measure the recall and precision of our retrieval on the manually annotated dataset. 

In [ ]:

# Reload the model — weights are already cached locally, so this should be fast, no re-download
model = BGEM3FlagModel('BAAI/bge-m3', use_fp16=True, device='cuda:0', max_seq_len=2048, use_cache=True)

# Redefine the query and its embeddings — wiped by the restart
query = "quantization of neural network weights for efficient inference"
q_out = model.encode([query], return_dense=True, return_sparse=True)
query_dense = q_out["dense_vecs"][0]
query_sparse = q_out["lexical_weights"][0]

# Redefine the fusion helper — also wiped
def rrf_fuse(*score_arrays, k=60):
    fused = np.zeros(len(score_arrays[0]))
    for scores in score_arrays:
        ranks = np.argsort(np.argsort(-scores))
        fused += 1.0 / (k + ranks + 1)
    return fused

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 30 files:   0%|          | 0/30 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

In [13]:
golden = pd.read_parquet("data/interim/quantization_golden_set.parquet")
golden = golden.dropna(subset=["label"])
golden["label"] = golden["label"].astype(int)
print(golden["label"].value_counts())

label
0    61
1    24
Name: count, dtype: int64


In [14]:
g_texts = (golden["title"] + ". " + golden["abstract"]).tolist()
g_out = model.encode(g_texts, return_dense=True, return_sparse=True)
g_dense, g_sparse = g_out["dense_vecs"], g_out["lexical_weights"]

g_dense_scores = g_dense @ query_dense
g_sparse_scores = np.array([model.compute_lexical_matching_score(query_sparse, s) for s in g_sparse])
g_hybrid_scores = rrf_fuse(g_dense_scores, g_sparse_scores)

In [15]:
def evaluate_ranking(scores, labels, name):
    order = np.argsort(-scores)
    sorted_labels = np.array(labels)[order]
    n_pos = sorted_labels.sum()

    cum_hits = np.cumsum(sorted_labels)
    precision_at_k = cum_hits / (np.arange(len(sorted_labels)) + 1)
    recall_at_k = cum_hits / n_pos
    ap = precision_at_k[sorted_labels == 1].mean()   # average precision

    print(f"--- {name} ---  ({n_pos} true positives in golden set)")
    for k in [10, 20, int(n_pos)]:
        if k <= len(sorted_labels):
            print(f"  P@{k}: {precision_at_k[k-1]:.2f}   R@{k}: {recall_at_k[k-1]:.2f}")
    print(f"  Average Precision: {ap:.3f}")
    return ap

ap_dense  = evaluate_ranking(g_dense_scores,  golden["label"].values, "Dense only")
ap_hybrid = evaluate_ranking(g_hybrid_scores, golden["label"].values, "Hybrid (RRF)")

--- Dense only ---  (24 true positives in golden set)
  P@10: 0.70   R@10: 0.29
  P@20: 0.65   R@20: 0.54
  P@24: 0.67   R@24: 0.67
  Average Precision: 0.732
--- Hybrid (RRF) ---  (24 true positives in golden set)
  P@10: 0.80   R@10: 0.33
  P@20: 0.75   R@20: 0.62
  P@24: 0.75   R@24: 0.75
  Average Precision: 0.801


# Adding a LLM for classifying papers after hybrid search

Let's add a smal local LLMs to calssify each paper to increase precision. 

In [13]:
corrections = {
    "2412.10319": (1, 0, "SCBench — benchmark covering 8 categories of long-context methods; quantization is one of many, not the paper's own contribution"),
    "2409.00592": (1, 0, 'Hyper-Compression — paper explicitly states its method is "substantially different from... quantization"; quantization is only a comparison baseline'),
    "2502.01330": (1, 0, "Accelerating Linear RNNs — core method is unstructured sparsity (title); quantization is a minor deployment detail on one hardware target"),
    "2508.19432": (1, 0, "Quantized but Deceptive? — audits truthfulness of already-quantized models, doesn't propose a quantization method (same pattern as the already-correctly-excluded \"Quality Is Not a Safety Proxy\" paper)"),
    "2508.14000": (1, 0, "Formal Algorithms for Model Efficiency — formalizes pruning+quantization+distillation+PEFT together; quantization is one of several methods covered"),
    "1902.00153": (1, 0, "Deep Triplet Quantization — \"quantization\" here means discretizing representations into binary hash codes for retrieval (VQ-style), not weight/activation precision reduction"),
    "2009.13108": (0, 1, "NITI — trains with integer-only arithmetic throughout (8-bit); this is quantized training, a clear match"),
    "2308.10515": (0, 1, "QD-BEV — title literally says \"Quantization-aware\"; abstract confirms 4-bit weight / 6-bit activation QAT is the core method"),
}

golden_v2 = golden.copy()
for aid, (old, new, reason) in corrections.items():
    assert golden_v2.loc[golden_v2.arxiv_id == aid, "label"].values[0] == old, f"unexpected current label for {aid}"
    golden_v2.loc[golden_v2.arxiv_id == aid, "label"] = new

print(f"Old positive count: {(golden['label']==1).sum()}")
print(f"New positive count: {(golden_v2['label']==1).sum()}")
golden_v2.to_parquet("data/interim/quantization_golden_set_v2.parquet")

Old positive count: 24
New positive count: 20


In [9]:
golden = pd.read_parquet("data/interim/quantization_golden_set.parquet")
golden = golden.dropna(subset=["label"])
golden["label"] = golden["label"].astype(int)
print(golden["label"].value_counts())

label
0    61
1    24
Name: count, dtype: int64


In [14]:
few_shot = pd.concat([
    golden[golden["label"] == 1].sample(3, random_state=42),
    golden[golden["label"] == 0].sample(3, random_state=42),
])
eval_set = golden.drop(few_shot.index)   # never score the classifier on its own few-shot examples

few_shot_block = "\n".join(
    f'Title: {r["title"]}\nAbstract: {r["abstract"][:400]}\nAnswer: {"yes" if r["label"]==1 else "no"}\n'
    for _, r in few_shot.iterrows()
)

In [15]:
llm_name = "Qwen/Qwen2.5-7B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(llm_name)
llm = AutoModelForCausalLM.from_pretrained(
    llm_name, device_map="cuda:0",
    quantization_config=TorchAoConfig(Int8WeightOnlyConfig()),
    torch_dtype="auto",
)

OutOfMemoryError: CUDA out of memory. Tried to allocate 7.64 GiB. GPU 0 has a total capacity of 15.44 GiB of which 6.48 GiB is free. Including non-PyTorch memory, this process has 8.71 GiB memory in use. Of the allocated memory 8.13 GiB is allocated by PyTorch, and 255.48 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

RUBRIC = """You are screening arXiv papers for a literature review on: quantization of neural network
weights/activations — techniques that reduce numerical precision for efficient inference
(e.g. INT8/INT4/binary/ternary weights, post-training quantization (PTQ), quantization-aware
training (QAT), mixed-precision inference).

Answer "no" when quantization (in the sense above) is not the paper's own technique or main
contribution — including these look-alikes:
- "Uncertainty quantification": measuring model confidence — an unrelated subfield.
- "Vector quantization" / discrete codebooks for representation learning (e.g. VQ-VAE-style methods):
  discretizes representations, a different technique from weight/activation precision reduction.
- Papers that merely evaluate or use already-quantized models as a side topic (e.g. studying safety
  or robustness "under quantization") without proposing or analyzing a quantization method itself.

Missing a real quantization paper is worse than including a borderline one — a human reviews the
results afterward. If genuinely uncertain after considering the paper's core contribution, answer yes.
"""

In [17]:
def classify_paper(title, abstract):
    prompt = f"""{RUBRIC}
Examples:
{few_shot_block}
Now classify this paper. First, in one short sentence, name the paper's core technique. Then answer yes or no.

Title: {title}
Abstract: {abstract[:800]}

Reasoning:"""

    messages = [{"role": "user", "content": prompt}]
    inputs = tokenizer.apply_chat_template(
        messages, add_generation_prompt=True, return_tensors="pt", return_dict=True
    ).to(llm.device)
    output = llm.generate(**inputs, max_new_tokens=30, do_sample=False)   # bumped from 5 — reasoning needs room
    text = tokenizer.decode(output[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip().lower()
    return text.rstrip().rstrip(".").endswith("yes")   # answer now comes at the END, since reasoning comes first

In [18]:
eval_set = eval_set.copy()
eval_set["llm_predicted"] = [classify_paper(r["title"], r["abstract"]) for _, r in tqdm(eval_set.iterrows(), total=len(eval_set))]

tp = ((eval_set["llm_predicted"]) & (eval_set["label"] == 1)).sum()
fp = ((eval_set["llm_predicted"]) & (eval_set["label"] == 0)).sum()
fn = ((~eval_set["llm_predicted"]) & (eval_set["label"] == 1)).sum()
print(f"Precision: {tp/(tp+fp):.2f}   Recall: {tp/(tp+fn):.2f}   (n={len(eval_set)})")

  0%|          | 0/79 [00:00<?, ?it/s]

Precision: 0.80   Recall: 0.38   (n=79)


# Conclusion

It seems that no matter how I work around the LLM for classifying papers, the precisions increases but the recal dramatically decreases compared to the hybrid search.Let's move on with the OCR and RAG building to serve a local LLM on how to implement quantieation.
 